# 🚀 Antigravity Video Batch Renderer (FHD 1080p Super-Fast & 4K UHD)
Studio Otomatis Produksi Video Loop Motion Graphics (Seamless Loop 30 FPS) untuk Adobe Stock & Freepik.

### ⚡ Fitur Utama:
1. **Super Cepat Full HD (1080p)**: Hanya **~1 Menit per video** (hemat waktu 75%!).
2. **4K UHD (2160p)**: Kualitas Ultra High Definition (~5 Menit per video).
3. **Realtime Stream Auto-Download**: Setiap 1 video selesai langsung otomatis ter-download ke PC/Laptop tanpa takut disconnect!

## ⚙️ Step 1: Siapkan Engine Renderer (NVIDIA GPU & Google Chrome)

In [ ]:
import os, subprocess, shutil
os.chdir('/content')

print("⏳ [1/3] Memasang Google Chrome & Library GPU...")
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i /tmp/chrome.deb > /dev/null 2>&1 || apt-get install -fy > /dev/null 2>&1
!apt-get install -y ffmpeg libgbm-dev libnss3 libasound2 zip > /dev/null 2>&1
!rm -f /tmp/chrome.deb

print("⏳ [2/3] Mengunduh skrip renderer FHD/4K terbaru dari GitHub...")
if os.path.exists('/content/shadergradientpaper'):
    shutil.rmtree('/content/shadergradientpaper', ignore_errors=True)

!git clone https://github.com/consistmaker/shadergradientpaper.git /content/shadergradientpaper

print("⏳ [3/3] Menyiapkan modul WebGL & Puppeteer...")
%cd /content/shadergradientpaper
!npm install --legacy-peer-deps > /dev/null 2>&1
!npm install puppeteer-core > /dev/null 2>&1
!npm run build
%cd /content

print("\n✅ RENDER ENGINE SIAP 100%! SILAKAN LANJUT KE STEP 2 DI BAWAH.")

## 📥 Step 2: Masukkan Resep JSON dari Live Studio
1. Buka Studio Web Previewer di browser Anda: 👉 **[https://shadergradientpaper.vercel.app](https://shadergradientpaper.vercel.app)**
2. Atur warna, slider, dan shader sesuka Anda.
3. Klik tombol **Export Batch** -> Pilih **`Full HD (1080p)`** (Super Cepat ~1 Menit) atau **`4K UHD (2160p)`** -> Klik **Copy JSON**.
4. Paste JSON di dalam tanda kutip `RECIPE_JSON` di bawah ini, lalu jalankan Cell ini!

In [ ]:
import json
import os

# PASTE JSON RESEP DARI WEB (https://shadergradientpaper.vercel.app) DI SINI:
RECIPE_JSON = '''
{
  "metadata": {
    "targetResolution": "1920x1080 (Full HD - Fast Render)",
    "resolutionWidth": 1920,
    "resolutionHeight": 1080,
    "targetFps": 30,
    "loopDurationSeconds": 10,
    "isSeamlessLoop": true,
    "batchMode": "manual_queue"
  },
  "manualQueueList": [
    {
      "index": 1,
      "id": "item_1",
      "name": "Paper: mesh-gradient (#e0eaff)",
      "engine": "paper",
      "config": {
        "shaderType": "mesh-gradient",
        "color1": "#e0eaff",
        "color2": "#241d9a",
        "color3": "#f75092",
        "color4": "#9f50d3",
        "speed": 1.0,
        "distortion": 0.8,
        "swirl": 0.1
      }
    }
  ],
  "totalVideosInQueue": 1
}
'''

with open('/content/render_recipe.json', 'w') as f:
    f.write(RECIPE_JSON.strip())

recipe = json.loads(RECIPE_JSON)
print(f"🎯 Mode Render: {recipe['metadata'].get('batchMode', 'manual_queue').upper()}")
print(f"🎬 Target Resolusi: {recipe['metadata'].get('targetResolution', '1920x1080 (Full HD)')} @ {recipe['metadata'].get('targetFps', 30)} FPS")
print("✅ Resep JSON berhasil disimpan! Siap dirender di Step 3.")

## 🚀 Step 3: Eksekusi GPU Render (Auto-Download Video Langsung Tiap Selesai 1 Video)

In [ ]:
import subprocess, time, os, glob
from google.colab import files

output_dir = '/content/output_4k_videos'
os.makedirs(output_dir, exist_ok=True)
downloaded_files = set()

print("🚀 Memulai GPU Render Engine (FHD / 4K Auto-Detect)...")

process = subprocess.Popen(
    ['node', '/content/shadergradientpaper/headless_renderer.cjs'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

while True:
    line = process.stdout.readline()
    if line:
        print(line, end='')
        if '✅ Success' in line and 'Render:' in line:
            time.sleep(1)
            current_videos = glob.glob(f"{output_dir}/*.mp4")
            for v_path in current_videos:
                if v_path not in downloaded_files and os.path.exists(v_path):
                    file_size = (os.path.getsize(v_path) / (1024 * 1024))
                    print(f"   📥 [AUTO-DOWNLOAD] Mengunduh: {os.path.basename(v_path)} ({file_size:.2f} MB)...")
                    try:
                        files.download(v_path)
                        downloaded_files.add(v_path)
                    except Exception as e:
                        pass

    if process.poll() is not None:
        for remaining in process.stdout.readlines():
            print(remaining, end='')
        break

print(f"\n🎉 SELESAI! Seluruh video telah berhasil dirender dan terunduh ke laptop/PC Anda.")